In [1]:
# Block 1: Load Cleaned Data for Cross-Validation
import pandas as pd
import numpy as np

print(" Starting Cross-Table Validation...\n")

df_emp = pd.read_csv("D:\HR Project\HR data\cleaned\Dim_Employees_Cleaned.csv")
df_roles = pd.read_csv("D:\HR Project\HR data\cleaned\Dim_Job_Roles_Cleaned.csv")
df_att = pd.read_csv("D:\HR Project\HR data\cleaned\Fact_Attendance_Cleaned.csv")
df_comp = pd.read_csv("D:\HR Project\HR data\cleaned\Fact_Compensation_Cleaned.csv")
df_eng = pd.read_csv("D:\HR Project\HR data\cleaned\Fact_Engagement_Cleaned.csv")
df_perf = pd.read_csv("D:\HR Project\HR data\cleaned\Fact_Performance_Cleaned.csv")

# Convert date columns back to datetime (since CSV saving converts them to strings)
df_emp['exit_date'] = pd.to_datetime(df_emp['exit_date'])
df_perf['review_date'] = pd.to_datetime(df_perf['review_date'])
df_eng['survey_date'] = pd.to_datetime(df_eng['survey_date'])

print(" Cleaned Data Loaded Successfully.")

 Starting Cross-Table Validation...

 Cleaned Data Loaded Successfully.


In [ ]:
# Block 2: Salary Range Validation (The x8 Multiplier Trap)
print(" Checking: Are salaries within the official Job Role limits?")

# Join Compensation with Employees to get role_id
comp_emp = df_comp.merge(df_emp[['employee_id', 'role_id']], on='employee_id', how='left')

# Join with Job Roles to get min_salary and max_salary
comp_full = comp_emp.merge(df_roles[['role_id', 'job_title', 'min_salary', 'max_salary']], on='role_id', how='left')

# Flag salaries that are suspiciously high (e.g., higher than the max_salary of their role)
# We add a small buffer (e.g., 20% over max) for legitimate raises, anything above is a major outlier
df_comp['salary_out_of_range_flag'] = comp_full['monthly_salary_egp'] > (comp_full['max_salary'] * 1.2)

out_of_range_count = df_comp['salary_out_of_range_flag'].sum()
print(f"   ->  Found {out_of_range_count:,} records where the salary wildly exceeds the job role maximum!")
print("✅ Salary Range Validation complete.")

 Checking: Are salaries within the official Job Role limits?
   ->  Found 2,616 records where the salary wildly exceeds the job role maximum!
✅ Salary Range Validation complete.


In [4]:
comp_full

,compensation_id,employee_id,effective_date,monthly_salary_egp,annual_bonus_egp,stock_options,negative_bonus_flag,salary_outlier_flag,role_id,job_title,min_salary,max_salary
0,1,1,2022-10-28,9794.75,5368.97,0,False,False,57,Jr. Operations Analyst,8000.0,15000.0
1,2,1,2023-10-08,9890.30,1336.69,0,False,False,57,Jr. Operations Analyst,8000.0,15000.0
2,3,1,2024-04-19,11847.32,5345.91,0,False,False,57,Jr. Operations Analyst,8000.0,15000.0
3,4,1,2025-07-07,12554.45,4275.90,0,False,False,57,Jr. Operations Analyst,8000.0,15000.0
4,5,2,2020-04-09,83488.62,0.00,179,False,False,63,Operations Lead,60000.0,100000.0
...,...,...,...,...,...,...,...,...,...,...,...,...
67561,67562,14999,2024-01-07,67647.85,47438.60,195,False,False,60,Senior Operations Analyst,25000.0,60000.0
67562,67563,14999,2025-01-23,79914.95,9950.59,59,False,False,60,Senior Operations Analyst,25000.0,60000.0
67563,67564,15000,2023-07-05,88043.02,67019.28,235,False,False,9,Engineering Manager,60000.0,100000.0
67564,67565,15000,2024-03-07,100408.04,47978.75,61,False,False,9,Engineering Manager,60000.0,100000.0


In [6]:
df_comp

,compensation_id,employee_id,effective_date,monthly_salary_egp,annual_bonus_egp,stock_options,negative_bonus_flag,salary_outlier_flag,salary_out_of_range_flag
0,1,1,2022-10-28,9794.75,5368.97,0,False,False,False
1,2,1,2023-10-08,9890.30,1336.69,0,False,False,False
2,3,1,2024-04-19,11847.32,5345.91,0,False,False,False
3,4,1,2025-07-07,12554.45,4275.90,0,False,False,False
4,5,2,2020-04-09,83488.62,0.00,179,False,False,False
...,...,...,...,...,...,...,...,...,...
67561,67562,14999,2024-01-07,67647.85,47438.60,195,False,False,False
67562,67563,14999,2025-01-23,79914.95,9950.59,59,False,False,True
67563,67564,15000,2023-07-05,88043.02,67019.28,235,False,False,False
67564,67565,15000,2024-03-07,100408.04,47978.75,61,False,False,False


In [7]:
# Block 3: Timeline Consistency (Post-Exit Activity)
print("\n Checking: Are there activities recorded AFTER an employee resigned?")

# 1. Performance vs Exit Date
perf_emp = df_perf.merge(df_emp[['employee_id', 'exit_date']], on='employee_id', how='left')
# Check if review_date is strictly greater than exit_date
df_perf['post_exit_review_flag'] = perf_emp['review_date'] > perf_emp['exit_date']

# 2. Engagement vs Exit Date
eng_emp = df_eng.merge(df_emp[['employee_id', 'exit_date']], on='employee_id', how='left')
df_eng['post_exit_survey_flag'] = eng_emp['survey_date'] > eng_emp['exit_date']

print(f"   ->  Found {df_perf['post_exit_review_flag'].sum()} performance reviews AFTER exit dates.")
print(f"   ->  Found {df_eng['post_exit_survey_flag'].sum()} engagement surveys AFTER exit dates.")
print(" Timeline Validation complete.")


 Checking: Are there activities recorded AFTER an employee resigned?
   ->  Found 2191 performance reviews AFTER exit dates.
   ->  Found 958 engagement surveys AFTER exit dates.
 Timeline Validation complete.


In [8]:
perf_emp

,review_id,employee_id,review_date,performance_score,promoted_this_year,invalid_score_flag,exit_date
0,1,1,2022-11-19,2.0,0,False,NaT
1,2,1,2023-12-15,2.0,0,False,NaT
2,3,1,2024-12-14,2.0,1,False,NaT
3,4,1,2025-10-21,3.0,0,False,NaT
4,5,2,2020-10-26,3.0,0,False,NaT
...,...,...,...,...,...,...,...
67561,67562,14999,2025-01-06,5.0,1,False,NaT
67562,67563,14999,2025-11-14,5.0,1,False,NaT
67563,67564,15000,2023-12-03,4.0,0,False,NaT
67564,67565,15000,2024-12-10,4.0,0,False,NaT


In [9]:
df_perf

,review_id,employee_id,review_date,performance_score,promoted_this_year,invalid_score_flag,post_exit_review_flag
0,1,1,2022-11-19,2.0,0,False,False
1,2,1,2023-12-15,2.0,0,False,False
2,3,1,2024-12-14,2.0,1,False,False
3,4,1,2025-10-21,3.0,0,False,False
4,5,2,2020-10-26,3.0,0,False,False
...,...,...,...,...,...,...,...
67561,67562,14999,2025-01-06,5.0,1,False,False
67562,67563,14999,2025-11-14,5.0,1,False,False
67563,67564,15000,2023-12-03,4.0,0,False,False
67564,67565,15000,2024-12-10,4.0,0,False,False


In [10]:
# Block 4: Referential Integrity (Orphan Records)
print("\n Checking: Are there employee IDs in Fact tables that do NOT exist in Dim_Employees?")

valid_ids = df_emp['employee_id'].unique()

orphan_att = ~df_att['employee_id'].isin(valid_ids)
orphan_comp = ~df_comp['employee_id'].isin(valid_ids)

# Flag them
df_att['orphan_record_flag'] = orphan_att
df_comp['orphan_record_flag'] = orphan_comp

print(f"   -> Found {orphan_att.sum()} orphan records in Attendance.")
print(f"   -> Found {orphan_comp.sum()} orphan records in Compensation.")
print(" Referential Integrity Validation complete.")


 Checking: Are there employee IDs in Fact tables that do NOT exist in Dim_Employees?
   -> Found 0 orphan records in Attendance.
   -> Found 0 orphan records in Compensation.
 Referential Integrity Validation complete.


In [11]:
# Block 5: Exporting the Curated Layer
print("\n Exporting the Analysis-Ready (Curated) Data for Tableau/SQL...")

df_att.to_csv("Fact_Attendance_Curated.csv", index=False)
df_comp.to_csv("Fact_Compensation_Curated.csv", index=False)
df_eng.to_csv("Fact_Engagement_Curated.csv", index=False)
df_perf.to_csv("Fact_Performance_Curated.csv", index=False)

# Dimensions didn't change in this phase, but we can rename them to keep naming conventions consistent
df_emp.to_csv("Dim_Employees_Curated.csv", index=False)
df_roles.to_csv("Dim_Job_Roles_Curated.csv", index=False)

print("Data Pipeline Complete. Ready for Tableau & SQL!")


 Exporting the Analysis-Ready (Curated) Data for Tableau/SQL...
Data Pipeline Complete. Ready for Tableau & SQL!
